# v1 Baseline - Ridge Regression

Establishes the linear baseline for v1. All training logic is in models/train_v1.py. This notebook inspects results and tunes the alpha hyperparameter across a small grid.

Sets the baseline MAE for other models.

In [5]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../../")))

import mlflow
import pandas as pd
import matplotlib.pyplot as plt

from src.models.train_v1 import train_ridge, load_v1_splits
from src.features.feature_columns import SENSING_FEATURES, TARGET

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("mental_health_prediction")

print(f"MLflow tracking URI: {mlflow.get_tracking_uri()}")
print(f"Features: {len(SENSING_FEATURES)}")

MLflow tracking URI: http://localhost:5000
Features: 22


In [6]:
train, val, test = load_v1_splits()
print(f"Train: {len(train):,} rows | Val: {len(val):,} | Test: {len(test):,}")
print(f"Label mean — Train: {train[TARGET].mean():.1f} | "
      f"Val: {val[TARGET].mean():.1f} | Test: {test[TARGET].mean():.1f}")
print(f"NaN counts in train features:")
print(train[SENSING_FEATURES].isna().sum()[train[SENSING_FEATURES].isna().sum() > 0])

Train: 6,940 rows | Val: 1,610 | Test: 1,381
Label mean — Train: 64.8 | Val: 64.1 | Test: 63.6
NaN counts in train features:
label_lag1                          206
other_playing_duration_ep_0_mean    409
other_playing_duration_ep_0_std     410
dtype: int64


In [7]:
alphas = [0.1, 1.0, 10.0, 100.0]
results = []

for alpha in alphas:
    _, _, _, metrics = train_ridge(
        alpha=alpha,
        run_name=f"v1_ridge_alpha_{alpha}"
    )
    results.append({"alpha": alpha, **metrics})

results_df = pd.DataFrame(results)
print(results_df[["alpha", "train_mae", "val_mae", "test_mae"]].to_string(index=False))

2026/04/12 23:56:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v1_ridge_alpha_0.1 at: http://localhost:5000/#/experiments/2/runs/0cdc2efe540444b189f86b5dd1ae4dd0
🧪 View experiment at: http://localhost:5000/#/experiments/2
Ridge (alpha=0.1)
  train MAE: 10.0174  R2: 0.3364
  val   MAE: 9.8925   R2: 0.3789
  test  MAE: 10.5306   R2: 0.3184


2026/04/12 23:56:45 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v1_ridge_alpha_1.0 at: http://localhost:5000/#/experiments/2/runs/8b142dbb97fa4d2ca2b09be4b6872647
🧪 View experiment at: http://localhost:5000/#/experiments/2
Ridge (alpha=1.0)
  train MAE: 10.0174  R2: 0.3364
  val   MAE: 9.8926   R2: 0.3789
  test  MAE: 10.5306   R2: 0.3184


2026/04/12 23:56:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v1_ridge_alpha_10.0 at: http://localhost:5000/#/experiments/2/runs/662b0a4675764f0894519f8e5c7da9c4
🧪 View experiment at: http://localhost:5000/#/experiments/2
Ridge (alpha=10.0)
  train MAE: 10.0179  R2: 0.3364
  val   MAE: 9.8933   R2: 0.3789
  test  MAE: 10.531   R2: 0.3184


2026/04/12 23:56:59 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run v1_ridge_alpha_100.0 at: http://localhost:5000/#/experiments/2/runs/6b57c542f12f4d78b5f7f7331ab4392f
🧪 View experiment at: http://localhost:5000/#/experiments/2
Ridge (alpha=100.0)
  train MAE: 10.0229  R2: 0.3364
  val   MAE: 9.9013   R2: 0.3782
  test  MAE: 10.5355   R2: 0.3181
 alpha  train_mae  val_mae  test_mae
   0.1    10.0174   9.8925   10.5306
   1.0    10.0174   9.8926   10.5306
  10.0    10.0179   9.8933   10.5310
 100.0    10.0229   9.9013   10.5355


In [8]:
best = results_df.loc[results_df["val_mae"].idxmin()]
print(f"Best alpha by val_mae: {best['alpha']}")
print(f"  val MAE  : {best['val_mae']}")
print(f"  test MAE : {best['test_mae']}")
print(f"  val R2   : {best['val_r2']}")
print()
print("This is the Ridge baseline. LightGBM must beat this val_mae to justify.")

Best alpha by val_mae: 0.1
  val MAE  : 9.8925
  test MAE : 10.5306
  val R2   : 0.3789

This is the Ridge baseline. LightGBM must beat this val_mae to justify.
